In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, KBinsDiscretizer
from sklearn.feature_selection import mutual_info_classif, chi2, f_classif
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
from sklearn.metrics.pairwise import rbf_kernel
import warnings
warnings.filterwarnings("ignore")

# -------------------------
# Parameters
# -------------------------
B_BOOTSTRAP = 50
TAU = 0.6
MAX_WRAPPER_FINAL = 20
SEED = 42

# -------------------------
# Load dataset
# -------------------------
df = pd.read_csv("Student Engagement_Pilot_AUGMENTED.csv")

# Encode target -> convert to numpy array immediately
df['Class'] = LabelEncoder().fit_transform(df['Class'])
y = df['Class'].values   # <-- numpy array, avoids Series.reshape error
X = df.drop(columns=['Class'])

# Separate categorical and numeric
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

# One-hot encode categorical columns (keeps consistent column names)
if len(cat_cols) > 0:
    X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
else:
    X_encoded = X.copy()

# Convert boolean columns to integer (if any)
for col in X_encoded.columns:
    if X_encoded[col].dtype == 'bool':
        X_encoded[col] = X_encoded[col].astype(int)

# Scale numerical columns (only those that exist after encoding)
num_cols_after = [c for c in num_cols if c in X_encoded.columns]
if len(num_cols_after) > 0:
    scaler = StandardScaler()
    X_encoded[num_cols_after] = scaler.fit_transform(X_encoded[num_cols_after])

# Ensure X_encoded is a DataFrame
X_encoded = pd.DataFrame(X_encoded, columns=X_encoded.columns)

# -------------------------
# Scoring Functions
# -------------------------
def filter_scores(X_df, y_arr):
    X_np = X_df.values
    mi = mutual_info_classif(X_np, y_arr, random_state=SEED)
    # chi2 requires non-negative; use abs() as a pragmatic workaround
    chi2_scores, _ = chi2(np.abs(X_np), y_arr)
    f_scores, _ = f_classif(X_np, y_arr)
    return (mi + chi2_scores + f_scores) / 3.0

def embedded_scores(X_df, y_arr):
    X_np = X_df.values
    lasso = LassoCV(cv=5, random_state=SEED).fit(X_np, y_arr)
    lasso_scores = np.abs(lasso.coef_)
    rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X_np, y_arr)
    rf_scores = rf.feature_importances_
    return (lasso_scores + rf_scores) / 2.0

def interaction_scores(X_df, y_arr):
    """
    Interaction-aware score defined as average of:
      - PMI (mutual information between discretized single feature and y)
      - HSIC (kernel-based dependence between feature and y)

    This function returns a numpy array of length n_features.
    """
    n, p = X_df.shape
    scores_pmi = np.zeros(p, dtype=float)
    scores_hsic = np.zeros(p, dtype=float)

    # PMI: discretize each feature and compute MI(feature, y)
    kb = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
    for i in range(p):
        col = X_df.iloc[:, i].to_numpy().reshape(-1, 1)
        # handle constant columns gracefully
        if np.unique(col).shape[0] <= 1:
            scores_pmi[i] = 0.0
        else:
            try:
                col_disc = kb.fit_transform(col)
                # mutual_info_classif expects 2D X; we pass the single discretized column
                scores_pmi[i] = mutual_info_classif(col_disc, y_arr, random_state=SEED).mean()
            except Exception:
                scores_pmi[i] = 0.0

    # HSIC: kernel-based dependence between each feature and y
    # Use RBF kernel for continuous features; compute kernel on y as well.
    # convert y to 2D
    Y = np.asarray(y_arr).reshape(-1, 1)
    # Use gamma = 1.0 / (2 * sigma^2). We use default sigma=1 -> gamma=0.5
    Ky = rbf_kernel(Y, Y, gamma=0.5)
    # center Ky (optional but not necessary since we'll normalize later)
    n_samples = Y.shape[0]
    H = np.eye(n_samples) - np.ones((n_samples, n_samples)) / n_samples
    Ky_centered = H.dot(Ky).dot(H)

    for i in range(p):
        Xi = X_df.iloc[:, i].to_numpy().reshape(-1, 1)
        # if constant, HSIC 0
        if np.unique(Xi).shape[0] <= 1:
            scores_hsic[i] = 0.0
            continue
        Kx = rbf_kernel(Xi, Xi, gamma=0.5)
        Kx_centered = H.dot(Kx).dot(H)
        # empirical HSIC (trace(Kx H Ky H) / (n-1)^2). we use (n)^2 denom as pragmatic
        try:
            hsic_val = np.trace(Kx_centered.dot(Ky_centered)) / ((n_samples - 1.0)**2 + 1e-12)
        except Exception:
            hsic_val = 0.0
        scores_hsic[i] = hsic_val

    # combine PMI + HSIC
    scores = (scores_pmi + scores_hsic) / 2.0
    return scores

# -------------------------
# DAHFS (Single Run)
# -------------------------
def dahfs_once(X_df, y_arr):
    # Step 1: compute the three families of scores
    f_scores = filter_scores(X_df, y_arr)       # filter-based
    e_scores = embedded_scores(X_df, y_arr)     # embedded-based
    i_scores = interaction_scores(X_df, y_arr)  # PMI+HSIC

    # Step 2: normalize each to [0,1]
    def norm_arr(a):
        a = np.asarray(a, dtype=float)
        rng = np.ptp(a)
        if rng == 0:
            return np.zeros_like(a)
        return (a - np.min(a)) / (rng + 1e-12)

    f_scores = norm_arr(f_scores)
    e_scores = norm_arr(e_scores)
    i_scores = norm_arr(i_scores)

    # Step 3: hybrid fusion (weights can be tuned)
    beta_F, beta_E, beta_I = 0.33, 0.33, 0.34
    fused = beta_F * f_scores + beta_E * e_scores + beta_I * i_scores

    # Step 4: redundancy pruning by correlation clusters (rho > 0.85)
    cols = X_df.columns.tolist()
    corr = X_df.corr().abs().values  # p x p matrix
    keep = np.ones(len(cols), dtype=bool)
    for i in range(len(cols)):
        if not keep[i]:
            continue
        for j in range(i+1, len(cols)):
            if keep[j] and corr[i, j] > 0.85:
                # keep feature with higher fused score
                if fused[i] >= fused[j]:
                    keep[j] = False
                else:
                    keep[i] = False
                    break

    selected_cols = [c for k, c in zip(keep, cols) if k]
    return selected_cols, fused[keep]

# -------------------------
# Stability Selection over bootstrap runs
# -------------------------
def dahfs_stability(X_df, y_arr, B=B_BOOTSTRAP, tau=TAU, max_final=MAX_WRAPPER_FINAL):
    feature_counts = pd.Series(0.0, index=X_df.columns)

    print(f"Running DAHFS stability selection: B={B}, tau={tau}, max_final={max_final}")
    for b in range(B):
        if (b+1) % 10 == 0:
            print(f"  Bootstrap iteration {b+1}/{B}")
        X_res, y_res = resample(X_df, y_arr, replace=True, random_state=SEED + b)
        selected, _ = dahfs_once(X_res, y_res)
        # increment counts for selected features
        feature_counts[selected] += 1.0

    freq = feature_counts / B
    stable = freq[freq >= tau].sort_values(ascending=False)

    if len(stable) == 0 and len(freq) > 0:
        print(f"No features met tau={tau}. Selecting top {max_final} by frequency.")
        stable = freq.sort_values(ascending=False).head(max_final)
    elif len(stable) > max_final:
        print(f"More than {max_final} features met tau={tau}. Truncating to top {max_final}.")
        stable = stable.head(max_final)

    return stable

# -------------------------
# Run the DAHFS stability procedure
# -------------------------
stable_features = dahfs_stability(X_encoded, y, B=B_BOOTSTRAP, tau=TAU, max_final=MAX_WRAPPER_FINAL)

print("\nFinal Stable Features Selected (DAHFS):")
print(stable_features)